# Simple Optimization — lifetime_us + velocity_cm_us

Jointly optimizes two physics parameters using gradient descent.

**Setup**
- Ground truth: `lifetime_us = 10 000 μs`, `velocity_cm_us = 0.160 cm/μs`
- Random starting points drawn from N(GT, 20% σ)
- 2 optimizers: Adam, AdamW (implemented in JAX, no optax needed)
- 3 loss functions: Sobolev (spectral H⁻²), MSE, L1

**Running the optimization**

```bash
python run_opt.py [--output-dir results] [--n-tries 10] [--seed 42]
```

**This notebook** loads the saved pickle and produces plots — run the cells below after `run_opt.py` completes.

In [ ]:
# =============================================================================
# LOAD RESULTS  — run this cell to plot without re-running the full simulation
# =============================================================================
import pickle, os
import numpy as np
import matplotlib.pyplot as plt

os.makedirs('plots', exist_ok=True)

with open('results/optimization_results.pkl', 'rb') as f:
    d = pickle.load(f)

all_results       = d['all_results']
starts_n          = d['starts_n']
GT_LIFETIME_US    = d['GT_LIFETIME_US']
GT_VELOCITY_CM_US = d['GT_VELOCITY_CM_US']
N_STEPS           = d['N_STEPS']
N_TRIES           = d['N_TRIES']
NOISE_FRAC        = d['NOISE_FRAC']

OPT_COLORS = {'SGD': 'steelblue', 'Adam': 'darkorange', 'AdamW': 'forestgreen'}

print('Loaded results/optimization_results.pkl')
print('  Loss types :', list(all_results.keys()))
print('  Optimizers :', list(next(iter(all_results.values())).keys()))

In [ ]:
# =============================================================================
# PLOT: 2D TRAJECTORIES — one panel per loss type
# =============================================================================

loss_names = list(all_results.keys())
fig, axes = plt.subplots(1, len(loss_names), figsize=(10 * len(loss_names), 8))
if len(loss_names) == 1:
    axes = [axes]

for ax, loss_name in zip(axes, loss_names):
    for opt_name, trajs in all_results[loss_name].items():
        color = OPT_COLORS[opt_name]
        for i, traj in enumerate(trajs):
            lts  = [p[0] for p in traj]
            vels = [p[1] for p in traj]
            n = len(lts) - 1
            for s in range(n):
                alpha = 0.15 + 0.55 * (s / max(n - 1, 1))
                ax.plot(lts[s:s+2], vels[s:s+2], '-', color=color, alpha=alpha, lw=1.2)
            ax.plot(lts[0], vels[0], 'o', color=color, ms=7, alpha=0.85,
                    label=opt_name if i == 0 else None)
            ax.plot(lts[-1], vels[-1], 's', color=color, ms=5, alpha=0.85,
                    markeredgecolor='white', markeredgewidth=0.5)

    ax.plot(GT_LIFETIME_US, GT_VELOCITY_CM_US, '*', color='tomato', ms=18, zorder=6,
            label='Ground truth')
    ax.axvline(GT_LIFETIME_US, color='tomato', ls='--', lw=0.8, alpha=0.4)
    ax.axhline(GT_VELOCITY_CM_US, color='tomato', ls='--', lw=0.8, alpha=0.4)
    ax.set_xlabel('lifetime_us (μs)', fontsize=12)
    ax.set_ylabel('velocity_cm_us (cm/μs)', fontsize=12)
    ax.set_title(
        f'{loss_name} loss\n'
        f'{N_TRIES} starts × {N_STEPS} steps, ±{int(NOISE_FRAC*100)}% noise  |  '
        f'circles = start, squares = end',
        fontsize=11)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.25)

plt.tight_layout()
fig.savefig('plots/optimization_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# PLOT: LOSS CURVES — one panel per loss type  (mean ± std over N_TRIES)
# =============================================================================

loss_names = list(all_results.keys())
fig, axes = plt.subplots(1, len(loss_names), figsize=(9 * len(loss_names), 4))
if len(loss_names) == 1:
    axes = [axes]

steps = np.arange(N_STEPS + 1)

for ax, loss_name in zip(axes, loss_names):
    for opt_name, trajs in all_results[loss_name].items():
        color = OPT_COLORS[opt_name]
        loss_matrix = np.array([[p[2] for p in traj[1:]] for traj in trajs])
        mean = np.mean(loss_matrix, axis=0)
        std  = np.std(loss_matrix, axis=0)
        ax.semilogy(steps, mean, color=color, lw=2, label=opt_name)
        ax.fill_between(steps, np.maximum(mean - std, 1e-15), mean + std,
                        color=color, alpha=0.15)
    ax.set_xlabel('Optimisation step', fontsize=12)
    ax.set_ylabel('Loss value', fontsize=12)
    ax.set_title(f'{loss_name} loss  (mean ± 1σ over {N_TRIES} tries)', fontsize=11)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.25, which='both')

plt.tight_layout()
fig.savefig('plots/optimization_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()